# Esempio di analisi — Divorzio in Italia

Questo notebook mostra come usare i CSV di questo repository per rispondere a tre domande comuni:

1. Come è cambiata l'età media al divorzio in Italia?
2. Quanto dura un matrimonio prima del divorzio?
3. Chi presenta la domanda di divorzio: il marito o la moglie?

Il notebook usa solo la **libreria standard Python**. Una sezione opzionale in fondo mostra gli stessi esempi con `pandas` per chi preferisce.

> Fonte dati: ISTAT, via [open-data-divorzio-italia](https://github.com/divorziare/open-data-divorzio-italia). Licenza CC BY 4.0.

## 1. Età media al divorzio (marito, moglie)

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

REPO = Path('..').resolve()
CSV = REPO / 'data' / 'csv'

def read(path):
    with path.open(encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

marito = read(CSV / 'eta-al-divorzio-marito.csv')
moglie = read(CSV / 'eta-al-divorzio-moglie.csv')
print(f'Marito: {len(marito):,} righe')
print(f'Moglie: {len(moglie):,} righe')
print('Colonne:', list(marito[0].keys()))

In [ ]:
# Filtra le righe 'Totale' e calcola la distribuzione per anno
def totali_per_anno(rows, source_preferita='27_470_DF_DCIS_DIVORDEMCONG1_5'):
    out = defaultdict(int)
    for r in rows:
        if r.get('SOURCE') != source_preferita:
            continue
        # riga totale: tutte le classificazioni = 'Totale' o 'tutte le voci'
        if r.get('Classe di età del marito al divorzio', r.get('Classe di età della moglie al divorzio')) not in ('15 anni e più', 'Totale'):
            continue
        if r['VALUE']:
            out[int(r['TIME_PERIOD'])] += int(float(r['VALUE']))
    return dict(sorted(out.items()))

# Totale divorzi concessi per anno (dalla colonna marito, usata come conteggio)
for anno, tot in totali_per_anno(marito, source_preferita='27_470_DF_DCIS_DIVORDEMCONG1_5').items():
    print(f'{anno}: {tot:,} divorzi')

## 2. Durata media del procedimento di separazione

In [ ]:
proc = read(CSV / 'procedimenti-e-separazioni.csv')
durate = [r for r in proc if 'durata media' in r['Indicatore'].lower()]
for r in durate[:20]:
    print(f"{r['TIME_PERIOD']}: {r['Indicatore']} = {r['VALUE']}")

## 3. Chi presenta la domanda di divorzio?

In [ ]:
richiedente = read(CSV / 'coniuge-che-ha-presentato-la-domanda-di-divorzio-con-rito-giudiziale.csv')
print('Colonne:', list(richiedente[0].keys()))
print()
# Esempio: quante domande dal marito vs moglie nell'ultimo anno disponibile
ultimo_anno = max(int(r['TIME_PERIOD']) for r in richiedente if r['VALUE'])
print(f'Ultimo anno disponibile: {ultimo_anno}')

## Appendice — stessi esempi con pandas

Se hai `pandas` installato (`pip install pandas`), ecco come leggere gli stessi file:

In [ ]:
# import pandas as pd
# df = pd.read_csv(CSV / 'eta-al-divorzio-marito.csv')
# df['TIME_PERIOD'] = df['TIME_PERIOD'].astype(int)
# df[df['SOURCE'] == '27_470_DF_DCIS_DIVORDEMCONG1_5'].groupby('TIME_PERIOD')['VALUE'].sum()